# Nanochat GRPO Run on Kaggle

This notebook starts from the saved `d8_kaggle` SFT checkpoint and runs `scripts.chat_GRPO`.

Recommended Kaggle setup:

- GPU enabled (`2x Tesla T4` expected)
- Internet enabled
- Attach `nanochat-codes`
- Attach `nanochat-d8-sft-cache`

The notebook imports only the tokenizer and `chatsft_checkpoints/d8_kaggle` from the SFT cache, then writes GRPO checkpoints under `chatgrpo_checkpoints/d8_kaggle`.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path('/kaggle/input')

CODE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-codes'))
SFT_CACHE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-d8-sft-cache'))

assert CODE_INPUTS, 'Attach the nanochat-codes Kaggle dataset'
assert SFT_CACHE_INPUTS, 'Attach the nanochat-d8-sft-cache Kaggle dataset'

CODE_INPUT = CODE_INPUTS[0]
SFT_CACHE_INPUT = SFT_CACHE_INPUTS[0]

WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

required_cache_paths = [
    Path('tokenizer'),
    Path('chatsft_checkpoints') / 'd8_kaggle',
]
for rel_path in required_cache_paths:
    src_path = SFT_CACHE_INPUT / rel_path
    dst_path = WORK_CACHE / rel_path
    assert src_path.exists(), f'Missing required cache path in attached dataset: {src_path}'
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')

print('Code input:', CODE_INPUT)
print('SFT cache input:', SFT_CACHE_INPUT)
print('Working repo:', WORK_REPO)
print('Nanochat cache:', WORK_CACHE)
print('HF cache:', HF_CACHE)
subprocess.run(['df', '-h', '/kaggle/working'], check=False)
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)


Code input: /kaggle/input/datasets/yshuaiqin/nanochat-codes
SFT cache input: /kaggle/input/datasets/yshuaiqin/nanochat-d8-sft-cache
Working repo: /kaggle/working/nanochat
Nanochat cache: /kaggle/working/nanochat_cache
HF cache: /kaggle/working/huggingface_cache
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.1G   18G  11% /kaggle/working
2.1G	/kaggle/working/nanochat_cache


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Install Dependencies

Install the runtime packages the Kaggle image may not already include.


In [2]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'requests>=2.32.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
    'rustbpe>=0.1.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Dependencies installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 46.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.8 MB/s eta 0:00:00
Dependencies installed


## Configure Runtime

GRPO does not use GradScaler, so this notebook keeps the run in float32 for T4 stability.


In [3]:
env = os.environ.copy()
env['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
env['HF_HOME'] = str(HF_CACHE)
env['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
env['PYTHONUNBUFFERED'] = '1'

env['NANOCHAT_DTYPE'] = 'float32'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['TORCH_COMPILE_DISABLE'] = '1'
env['NANOCHAT_ADAMW_ALLREDUCE'] = '1'
env['NANOCHAT_SERIAL_OPTIM_COMM'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'
env['TORCH_NCCL_ASYNC_ERROR_HANDLING'] = '1'
env['NCCL_DEBUG'] = 'WARN'

os.environ.update(env)

import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## Validate Repo And SFT Cache

Check the active Kaggle working copy and attached SFT cache before starting training.


In [4]:
assert (WORK_REPO / 'scripts' / 'chat_GRPO.py').exists(), 'Missing scripts/chat_GRPO.py'
assert (WORK_REPO / 'scripts' / 'chat_ppo.py').exists(), 'Missing scripts/chat_ppo.py'
assert (WORK_REPO / 'scripts' / 'chat_post_eval.py').exists(), 'Missing scripts/chat_post_eval.py'
assert (WORK_REPO / 'scripts' / 'chat_web.py').exists(), 'Missing scripts/chat_web.py'
assert (WORK_REPO / 'nanochat' / 'checkpoint_manager.py').exists(), 'Missing nanochat/checkpoint_manager.py'

MODEL_TAG = 'd8_kaggle'
sft_dir = WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG
tokenizer_dir = WORK_CACHE / 'tokenizer'
assert sft_dir.exists(), f'Missing SFT checkpoint dir: {sft_dir}'
assert tokenizer_dir.exists(), f'Missing tokenizer dir: {tokenizer_dir}'
assert sorted(sft_dir.glob('model_*.pt')), f'No model_*.pt files found in {sft_dir}'
assert sorted(sft_dir.glob('meta_*.json')), f'No meta_*.json files found in {sft_dir}'

subprocess.check_call(
    [
        sys.executable, '-m', 'py_compile',
        'scripts/chat_GRPO.py',
        'scripts/chat_ppo.py',
        'scripts/chat_post_eval.py',
        'scripts/chat_web.py',
        'nanochat/checkpoint_manager.py',
    ],
    cwd=WORK_REPO,
    env=env,
)

print('Setup complete')
print('SFT checkpoints:', [p.name for p in sorted(sft_dir.glob('model_*.pt'))[-5:]])
print('Tokenizer files:', sorted(p.name for p in tokenizer_dir.iterdir()))


Setup complete
SFT checkpoints: ['model_004999.pt']
Tokenizer files: ['token_bytes.pt', 'tokenizer.pkl']


## GRPO Run

This run trains a reward model from synthetic GSM8K preferences, samples completion groups from the SFT policy, and optimizes group-relative advantages with a KL penalty to the SFT reference.


In [5]:
NPROC = 2 if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else 1
MODEL_TAG = 'd8_kaggle'
OVERWRITE_GRPO_CHECKPOINTS = True

if OVERWRITE_GRPO_CHECKPOINTS:
    shutil.rmtree(WORK_CACHE / 'chatgrpo_checkpoints' / MODEL_TAG, ignore_errors=True)
    shutil.rmtree(WORK_CACHE / 'chatgrpo_reward_checkpoints' / MODEL_TAG, ignore_errors=True)

grpo_args = [
    '-m', 'scripts.chat_GRPO',

    '--run=dummy',

    '--policy-source=sft',
    f'--policy-tag={MODEL_TAG}',
    '--reference-source=sft',
    f'--reference-tag={MODEL_TAG}',

    '--preference-source=gsm8k',
    '--max-train-examples=4096',
    '--max-val-examples=512',

    '--rm-epochs=2',
    '--rm-batch-size=4',
    '--rm-lr=1e-4',
    '--rm-train-backbone=0',

    '--grpo-steps=300',
    '--grpo-epochs=1',
    '--prompt-batch-size=2',
    '--group-size=4',
    '--device-batch-size=2',
    '--max-new-tokens=256',

    '--temperature=1.0',
    '--top-k=0',

    '--clip-epsilon=0.2',
    '--kl-beta=0.02',
    '--scale-rewards=1',

    '--embedding-lr=0.03',
    '--unembedding-lr=0.0008',
    '--matrix-lr=0.002',
    '--init-lr-frac=0.05',
    '--weight-decay=0.0',

    '--save-every=100',
]

if NPROC > 1:
    cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + grpo_args
else:
    cmd = [sys.executable] + grpo_args

print('Running command:')
print(' '.join(str(x) for x in cmd))
subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)

policy_dir = WORK_CACHE / 'chatgrpo_checkpoints' / MODEL_TAG
reward_dir = WORK_CACHE / 'chatgrpo_reward_checkpoints' / MODEL_TAG

print('GRPO policy checkpoints:', [p.name for p in sorted(policy_dir.glob('model_*.pt'))])
print('GRPO reward checkpoints:', [p.name for p in sorted(reward_dir.glob('reward_*.pt'))])


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_GRPO --run=dummy --policy-source=sft --policy-tag=d8_kaggle --reference-source=sft --reference-tag=d8_kaggle --preference-source=gsm8k --max-train-examples=4096 --max-val-examples=512 --rm-epochs=2 --rm-batch-size=4 --rm-lr=1e-4 --rm-train-backbone=0 --grpo-steps=300 --grpo-epochs=1 --prompt-batch-size=2 --group-size=4 --device-batch-size=2 --max-new-tokens=256 --temperature=1.0 --top-k=0 --clip-epsilon=0.2 --kl-beta=0.02 --scale-rewards=1 --embedding-lr=0.03 --unembedding-lr=0.0008 --matrix-lr=0.002 --init-lr-frac=0.05 --weight-decay=0.0 --save-every=100


[W614 05:00:58.922401493 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 05:01:09,807 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 05:01:09,807 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 05:01:09,809 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 05:01:09,809 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 05:01:11.291023207 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 05:01:11.291886216 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Saved GRPO checkpoints to /kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle and /kaggle/working/nanochat_cache/chatgrpo_reward_checkpoints/d8_kaggle
Step 101 | pg_loss: 0.015527 | kl_loss: 0.042702 | reward: 0.0712 | reward_group_std: 0.6403 | ref_kl: 0.0427 | ratio: 1.0000 | clipfrac: 0.0000
Step 102 | pg_loss: 0.228407 | kl_loss: 0.040121 | reward: -0.0992 | reward_group_std: 0.7297 | ref_kl: 0.0401 | ratio: 1.0000 | clipfrac: 0.0000
Step 103 | pg_loss: 0.086945 | kl_loss: 0.075146 | reward: 0.2432 | reward_group_std: 0.9076 | ref_kl: 0.0751 | ratio: 1.0000 | clipfrac: 0.0000
Step 104 | pg_loss: 0.097931 | kl_loss: 0.043320 | reward: 0.1865 | reward_group_std: 0.6355 | ref_kl: 0.0433 | ratio: 1.0000 | clipfrac: 0.0000
Step 105 | pg_loss: -0.023102 | kl_loss: 0.034656 | reward: -0.3618 | reward_group_std: 0.6457 | ref_kl: 0.0347 | ratio: 1.0000 | clipfrac: 0.0000
Step 106 | pg_loss: -0.075420 | kl_loss: 0.041793 | reward: -0.3026 | reward_group_std: 0.6230 | ref_kl: 0.0418

2026-06-14 05:48:13,502 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle/model_000200.pt
2026-06-14 05:48:13,502 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle/meta_000200.json


Saved GRPO checkpoints to /kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle and /kaggle/working/nanochat_cache/chatgrpo_reward_checkpoints/d8_kaggle
Step 201 | pg_loss: -0.071162 | kl_loss: 0.132478 | reward: 0.3884 | reward_group_std: 0.4620 | ref_kl: 0.1325 | ratio: 1.0000 | clipfrac: 0.0000
Step 202 | pg_loss: -0.072824 | kl_loss: 0.144040 | reward: 0.6574 | reward_group_std: 0.5803 | ref_kl: 0.1440 | ratio: 1.0000 | clipfrac: 0.0000
Step 203 | pg_loss: -0.095330 | kl_loss: 0.087726 | reward: 0.5761 | reward_group_std: 0.8003 | ref_kl: 0.0877 | ratio: 1.0000 | clipfrac: 0.0000
Step 204 | pg_loss: 0.054821 | kl_loss: 0.112379 | reward: 0.3609 | reward_group_std: 0.4765 | ref_kl: 0.1124 | ratio: 1.0000 | clipfrac: 0.0000
Step 205 | pg_loss: 0.108162 | kl_loss: 0.092377 | reward: 0.5050 | reward_group_std: 0.9097 | ref_kl: 0.0924 | ratio: 1.0000 | clipfrac: 0.0000
Step 206 | pg_loss: 0.127688 | kl_loss: 0.130413 | reward: 0.4737 | reward_group_std: 0.5732 | ref_kl: 0.1304 |

2026-06-14 06:06:15,984 - nanochat.checkpoint_manager - INFO - Saved model parameters to: /kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle/model_000300.pt
2026-06-14 06:06:15,985 - nanochat.checkpoint_manager - INFO - Saved metadata to: /kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle/meta_000300.json


Saved GRPO checkpoints to /kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle and /kaggle/working/nanochat_cache/chatgrpo_reward_checkpoints/d8_kaggle
GRPO policy checkpoints: ['model_000100.pt', 'model_000200.pt', 'model_000300.pt']
GRPO reward checkpoints: ['reward_000100.pt', 'reward_000200.pt', 'reward_000300.pt']


## Inspect GRPO Checkpoints

Confirm the policy and reward checkpoint folders before evaluation or saving.


In [6]:
families = [
    'chatgrpo_checkpoints',
    'chatgrpo_reward_checkpoints',
]
for family in families:
    checkpoint_dir = WORK_CACHE / family / MODEL_TAG
    print()
    print(family, checkpoint_dir)
    print('Exists:', checkpoint_dir.exists())
    if checkpoint_dir.exists():
        print('Files:', [p.name for p in sorted(checkpoint_dir.glob('*'))])

subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)



chatgrpo_checkpoints /kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle
Exists: True
Files: ['meta_000100.json', 'meta_000200.json', 'meta_000300.json', 'model_000100.pt', 'model_000200.pt', 'model_000300.pt']

chatgrpo_reward_checkpoints /kaggle/working/nanochat_cache/chatgrpo_reward_checkpoints/d8_kaggle
Exists: True
Files: ['reward_000100.pt', 'reward_000200.pt', 'reward_000300.pt']
4.9G	/kaggle/working/nanochat_cache


CompletedProcess(args=['du', '-sh', '/kaggle/working/nanochat_cache'], returncode=0)

## Comparison Eval

Compare the SFT checkpoint against the GRPO checkpoint on a small fixed evaluation slice.


In [7]:
RUN_COMPARISON_EVAL = True
EVAL_MAX_PROBLEMS = 100

if RUN_COMPARISON_EVAL:
    models = ['sft=sft:d8_kaggle']

    grpo_dir = WORK_CACHE / 'chatgrpo_checkpoints'
    if (grpo_dir / MODEL_TAG).exists():
        models.append(f'grpo=@{grpo_dir}:{MODEL_TAG}')

    post_eval_args = [
        '-m', 'scripts.chat_post_eval',
    ]
    for model in models:
        post_eval_args.extend(['--models', model])
    post_eval_args.extend([
        '--max-problems', str(EVAL_MAX_PROBLEMS),
        '--batch-size', '2',
    ])

    if NPROC > 1:
        cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + post_eval_args
    else:
        cmd = [sys.executable] + post_eval_args

    print('Running command:')
    print(' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)
else:
    print('Skipping comparison eval')


Running command:
torchrun --standalone --nproc_per_node=2 -m scripts.chat_post_eval --models sft=sft:d8_kaggle --models grpo=@/kaggle/working/nanochat_cache/chatgrpo_checkpoints:d8_kaggle --max-problems 100 --batch-size 2


[W614 06:06:22.061987783 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
2026-06-14 06:06:26,290 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 06:06:26,291 - datasets - INFO - JAX version 0.7.2 available.
2026-06-14 06:06:26,379 - datasets - INFO - TensorFlow version 2.19.0 available.
2026-06-14 06:06:26,380 - datasets - INFO - JAX version 0.7.2 available.


Autodetected device type: cuda


[W614 06:06:27.192900737 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W614 06:06:27.232179020 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


NCCL version 2.27.5+cuda12.9


...

Final: 18/100 (18.00%)
grpo | MMLU: 18.00%


2026-06-14 06:12:39,067 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 06:12:39,071 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 06:12:39,073 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 06:12:39,078 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-06-14 06:12:39,090 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-06-14 06:12:39,094 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k

Rank 0 | 0/50 (0.00%)
Rank 1 | 0/50 (0.00%)
Final: 0/100 (0.00%)
grpo | GSM8K: 0.00%


2026-06-14 06:13:51,552 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 06:13:51,557 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 06:13:51,575 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/openai_humaneval.py "HTTP/1.1 404 Not Found"
2026-06-14 06:13:51,576 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/datasets/openai/openai_humaneval/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-06-14 06:13:51,584 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/openai_humaneval/7dce6050a7d6d172f3cc5c32aa97f52fa1a2e544/README.md "HTTP/1.1 200 OK"
2026-06-14 06:13:51,604 - httpx - INFO

Rank 0 | 0/50 (0.00%)
Rank 1 | 0/50 (0.00%)
Final: 0/100 (0.00%)
grpo | HumanEval: 0.00%
Rank 0 | 47/50 (94.00%)
Rank 1 | 46/50 (92.00%)
Final: 93/100 (93.00%)
grpo | SpellingBee: 93.00%

Model | ARC-Easy | ARC-Challenge | MMLU   | GSM8K | HumanEval | SpellingBee | Mean  
------+----------+---------------+--------+-------+-----------+-------------+-------
sft   | 26.00%   | 29.00%        | 24.00% | 0.00% | 0.00%     | 95.00%      | 29.00%
grpo  | 26.00%   | 31.00%        | 18.00% | 0.00% | 0.00%     | 93.00%      | 28.00%

Ranking by mean score:
1. sft: 29.00%
2. grpo: 28.00%


## Inspect Saved Comparison Report

Display the report written by `scripts.chat_post_eval`.


In [8]:
report_path = WORK_CACHE / 'report' / 'chat-post-eval.md'
print(report_path)
print('Exists:', report_path.exists())
if report_path.exists():
    print(report_path.read_text())


/kaggle/working/nanochat_cache/report/chat-post-eval.md
Exists: True
## Chat Post Eval
timestamp: 2026-06-14 06:17:54

- tasks: ['ARC-Easy', 'ARC-Challenge', 'MMLU', 'GSM8K', 'HumanEval', 'SpellingBee']
- num_samples: 1
- temperature: 0.0000
- max_new_tokens: 512
- batch_size: 2
- max_problems: 100
- models: [{'label': 'sft', 'origin': 'sft', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 4999, 'resolved_meta_keys': ['model_config', 'step', 'user_config', 'val_bpb']}, {'label': 'grpo', 'origin': '/kaggle/working/nanochat_cache/chatgrpo_checkpoints', 'checkpoint_dir': '/kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle', 'model_tag': 'd8_kaggle', 'step': 300, 'resolved_meta_keys': ['model_config']}]
- results: [{'label': 'sft', 'ARC-Easy': 0.26, 'ARC-Challenge': 0.29, 'MMLU': 0.24, 'GSM8K': 0.0, 'HumanEval': 0.0, 'SpellingBee': 0.95}, {'label': 'grpo', 'ARC-Easy': 0.26, 'ARC-Challenge': 0.31, 'MMLU': 0.18, '

## Output Manifest

Write a compact manifest of the artifacts produced by this notebook.


In [9]:
manifest = {
    'model_tag': MODEL_TAG,
    'sft_checkpoint_dir': str(WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG),
    'grpo_checkpoint_dir': str(WORK_CACHE / 'chatgrpo_checkpoints' / MODEL_TAG),
    'grpo_reward_checkpoint_dir': str(WORK_CACHE / 'chatgrpo_reward_checkpoints' / MODEL_TAG),
    'report': str(WORK_CACHE / 'report' / 'chat-post-eval.md'),
}
manifest_path = Path('/kaggle/working/nanochat_grpo_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(manifest_path)
print(manifest_path.read_text())


/kaggle/working/nanochat_grpo_manifest.json
{
  "model_tag": "d8_kaggle",
  "sft_checkpoint_dir": "/kaggle/working/nanochat_cache/chatsft_checkpoints/d8_kaggle",
  "grpo_checkpoint_dir": "/kaggle/working/nanochat_cache/chatgrpo_checkpoints/d8_kaggle",
  "grpo_reward_checkpoint_dir": "/kaggle/working/nanochat_cache/chatgrpo_reward_checkpoints/d8_kaggle",
  "report": "/kaggle/working/nanochat_cache/report/chat-post-eval.md"
}


In [10]:
# Optional: save the GRPO checkpoint cache as a Kaggle Dataset.
import json

GRPO_MODEL_TAG = MODEL_TAG
GRPO_CACHE_DIR = Path('/kaggle/working/nanochat_d8_grpo_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-grpo-cache'
TITLE = 'nanochat d8 grpo cache'
VERSION_MESSAGE = f'Save {GRPO_MODEL_TAG} GRPO checkpoint cache'
UPLOAD_ACTION = 'create'  # use 'version' after the Dataset already exists

required_dirs = [
    WORK_CACHE / 'chatgrpo_checkpoints' / GRPO_MODEL_TAG,
    WORK_CACHE / 'chatgrpo_reward_checkpoints' / GRPO_MODEL_TAG,
    WORK_CACHE / 'tokenizer',
]
for required_dir in required_dirs:
    assert required_dir.exists(), f'Missing required directory: {required_dir}'

assert sorted((WORK_CACHE / 'chatgrpo_checkpoints' / GRPO_MODEL_TAG).glob('model_*.pt')), 'No GRPO policy checkpoints found'
assert sorted((WORK_CACHE / 'chatgrpo_reward_checkpoints' / GRPO_MODEL_TAG).glob('reward_*.pt')), 'No GRPO reward checkpoints found'

if GRPO_CACHE_DIR.exists():
    shutil.rmtree(GRPO_CACHE_DIR)
GRPO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for family in [
    'chatgrpo_checkpoints',
    'chatgrpo_reward_checkpoints',
]:
    shutil.copytree(WORK_CACHE / family, GRPO_CACHE_DIR / family)
shutil.copytree(WORK_CACHE / 'tokenizer', GRPO_CACHE_DIR / 'tokenizer')

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = GRPO_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(GRPO_CACHE_DIR)], check=False)

if UPLOAD_ACTION == 'create':
    cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(GRPO_CACHE_DIR),
        '--dir-mode', 'tar',
    ]
elif UPLOAD_ACTION == 'version':
    cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(GRPO_CACHE_DIR),
        '-m', VERSION_MESSAGE,
        '--dir-mode', 'tar',
    ]
else:
    raise ValueError("UPLOAD_ACTION must be 'create' or 'version'")

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


Folder size:
2.9G	/kaggle/working/nanochat_d8_grpo_cache
Running: kaggle datasets create -p /kaggle/working/nanochat_d8_grpo_cache --dir-mode tar
Starting upload for file chatgrpo_checkpoints.tar


100%|██████████| 1.41G/1.41G [00:13<00:00, 110MB/s]


Upload successful: chatgrpo_checkpoints.tar (1GB)
Starting upload for file chatgrpo_reward_checkpoints.tar


100%|██████████| 1.41G/1.41G [00:14<00:00, 101MB/s]


Upload successful: chatgrpo_reward_checkpoints.tar (1GB)
Starting upload for file tokenizer.tar


100%|██████████| 540k/540k [00:00<00:00, 1.98MB/s]


Upload successful: tokenizer.tar (540KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-grpo-cache
Done.
Dataset URL: https://www.kaggle.com/datasets/yshuaiqin/nanochat-d8-grpo-cache


## Serve the GRPO Chat Model

Start a local NanoChat server from the GRPO checkpoint.


In [ ]:
import time
import requests

SERVE_MODEL_TAG = MODEL_TAG
SERVER_PORT = 8000
BASE_URL = f'http://127.0.0.1:{SERVER_PORT}'
CHECKPOINT_DIR = WORK_CACHE / 'chatgrpo_checkpoints'

model_dir = CHECKPOINT_DIR / SERVE_MODEL_TAG
assert model_dir.exists(), f'Missing GRPO checkpoint directory: {model_dir}'
assert sorted(model_dir.glob('model_*.pt')), f'No model_*.pt files found in {model_dir}'
assert sorted(model_dir.glob('meta_*.json')), f'No meta_*.json files found in {model_dir}'

try:
    if server.poll() is None:
        print('Stopping existing NanoChat server...')
        server.terminate()
        server.wait(timeout=10)
        print('Stopped existing server.')
except NameError:
    pass
except Exception as exc:
    print('Could not stop existing server cleanly:', exc)
    try:
        server.kill()
        server.wait(timeout=10)
        print('Killed existing server.')
    except Exception:
        pass

messages = []

server_env = env.copy()
server_env['NANOCHAT_DISABLE_COMPILE'] = '1'
server_env['TORCH_COMPILE_DISABLE'] = '1'
server_env['OMP_NUM_THREADS'] = '1'

cmd = [
    sys.executable,
    '-m', 'scripts.chat_web',
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--model-tag', SERVE_MODEL_TAG,
    '--num-gpus', '1',
    '--port', str(SERVER_PORT),
]

print('Starting NanoChat server:')
print(' '.join(cmd))
server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f'Server process started with PID {server.pid}.')

SERVER_READY = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f'NanoChat server exited early with code {server.returncode}')
    try:
        response = requests.get(f'{BASE_URL}/health', timeout=2)
        if response.ok:
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f'NanoChat server is ready: {BASE_URL}')
else:
    print(f'NanoChat server is still loading. Wait a bit, then use: {BASE_URL}')


In [ ]:
import json
import requests

BASE = globals().get('BASE_URL', 'http://127.0.0.1:8000')
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({'role': 'user', 'content': prompt})

    payload = {
        'messages': messages,
        'temperature': temperature,
        'top_k': top_k,
        'max_tokens': max_tokens,
    }

    print('Assistant:', end=' ', flush=True)
    answer = ''

    with requests.post(f'{BASE}/chat/completions', json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith('data: '):
                continue

            data = json.loads(line[len('data: '):])
            if data.get('done'):
                break

            token = data.get('token', '')
            answer += token
            print(token, end='', flush=True)

    print()
    messages.append({'role': 'assistant', 'content': answer})
    return answer

print(f'Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.')
while True:
    prompt = input('\nYou: ')
    command = prompt.strip().lower()
    if command in {'q', 'quit', 'exit'}:
        break
    if command == 'reset':
        messages.clear()
        print('Chat history cleared.')
        continue
    ask(prompt)


In [ ]:
# Stop the NanoChat web server started by the serving cell.
import os
import subprocess
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat server killed.')
        stopped_any = True
    else:
        print(f'NanoChat server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `server` variable found.')

try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat server process found.')
